#### 1.Read data from the apache-logs.txt file

1.1 load and display the data

In [0]:
file_df =(
  spark.read.format('text')
  .load('/Volumes/dev/spark_db/datasets/spark_programming/data/apache-logs.txt')
)

display(file_df)


In [0]:
file_df.printSchema()

####2. Develop an strategy to extract the following fields
1. ip_address: It is the IP address of the site visitor.
2. visit_timestamp: It is the date and time of the site visit. Parse and format the timestamp to YYYY-MM-DD HH:MI:SS Z
3. visit_resource: Which resource from our website was accessed
4. referring_url: It is the clean URL of the referring website.

2.1 develop the regular experssion

In [0]:
log_reg = r'^(\S+) (\S+) (\S+) \[([\w:/]+\s[+\-]\d{4})\] "(\S+) (\S+) (\S+)" (\d{3}) (\S+) "(\S+)" "([^"]*)'

2.2 Apply regular expression to parse the record

In [0]:
from pyspark.sql.functions import regexp_extract

logs_df = (
    file_df.select(
        regexp_extract("value", log_reg, 1).alias("ip_address"),
        regexp_extract("value", log_reg, 4).alias("visit_timestamp"),
        regexp_extract('value', log_reg, 6).alias('visit_resource'),
        regexp_extract('value', log_reg, 10).alias('referring_url')
    )
)

logs_df.display()

In [0]:
prompt = """
You will be provided with an Apache log file record. It is an unstructured text record. 
Each record represents some information for our website visits, such as what is the IP address of the visitor, 
What is the date and time of the visit, which resource was requested, and the URL of the referring website? 
You are asked to parse the log file record and extract the following fields.
ip_address: It is the IP address of the site visitor.
visit_timestamp: It is the date and time of the site visit. Parse and format the timestamp to YYYY-MM-DD HH:MI:SS Z
visit_resource: Which resource from our website was accessed?
referring_url: It is the clean URL of the referring website. When the actual referring URL is not given, 
you can extract the URL from the user agent. For cleaning the URL, you should take the values only up to the domain extension, 
such as .com, .in, .uk, etc.
Give only the final answer in the JSON format.
Record:
"""

Develop an AI query expression

In [0]:
from pyspark.sql.functions import concat, col, lit, expr

result_df = (
    file_df.limit(20)
        .withColumn("prompt", concat(lit(prompt), col("value")))
        .withColumn("json_extract", expr("""
            ai_query(
                endpoint=> 'databricks-llama-4-maverick',
                request=> prompt,
                responseFormat=> 'struct<extract: struct<
                ip_address: string,
                visit_timestamp: string,
                visit_resource: string,
                referring_url: string
                >>')"""))
)

result_df.display()

Parse the JSON extract to individual columns

In [0]:
from pyspark.sql.functions import from_json, col, to_timestamp

extract_schema = "ip_address string, visit_timestamp string, visit_resource string, referring_url string"

final_result_df = (
    result_df.withColumn("json_extract", from_json(col("json_extract"), extract_schema))
             .selectExpr("json_extract.*")
             .withColumn("visit_timestamp", to_timestamp(col("visit_timestamp"), "yyyy-MM-dd HH:mm:ss Z"))
)

final_result_df.display()